# AirAdvisor: where the next claims are, and how to grow them profitably

One notebook, five questions. **PUBLIC DATA** parts use real, sourced data (`data/SOURCES.md`); **SIMULATED** parts use an invented
claim funnel (`src/simulate.py`, seed 42) to show the method. Simulated numbers are never AirAdvisor's.

Order to run first: `python src/download_data.py`, `python src/build_public.py`, `python src/simulate.py`, `python src/analysis.py`.
The heavy lifting lives in `src/` and `sql/`; this notebook shows and explains the results.

In [1]:
import sys, json
sys.path.insert(0, '../src')
import polars as pl
from common import sim_con, run_sql_file, PUB, DATA
import analysis as A
pl.Config.set_tbl_rows(12); pl.Config.set_tbl_cols(12); pl.Config.set_fmt_float('mixed')
con = sim_con()               # DuckDB views over data/simulated/*.parquet and data/public/*.csv
R = json.loads((DATA / 'output' / 'results.json').read_text(encoding='utf-8'))

## 1. Where are the next claims? (PUBLIC DATA)

Per departure country: passengers on each route (Eurostat 2024, UK CAA 2025) x share of flights 3h+ late or cancelled
(UK CAA 2025, routes between the UK and that country) x the EU261/UK261 amount for the route distance (OurAirports).
That is a **theoretical ceiling**: extraordinary circumstances remove the right to compensation and most passengers never claim.
The model then applies the share within the airline's control, the fee, the claim rate, the agency share, the payout rate and an obtainable share.

In [2]:
m = pl.read_csv(DATA / 'output' / 'market_ranking.csv')
print(f"Ceiling €{R['market']['ceiling_eur']/1e9:.2f}bn | TAM €{R['market']['tam_eur']/1e6:.0f}M | SAM €{R['market']['sam_eur']/1e6:.0f}M | SOM €{R['market']['som_eur']/1e6:.1f}M")
m.select('rank', 'country_name', (pl.col('departing_pax')/1e6).round(1).alias('departures_M'),
         ((pl.col('p_late3') + pl.col('p_cancel'))*100).round(2).alias('disrupted_%'), pl.col('avg_amount_eur').round(0),
         (pl.col('som_eur')/1e6).round(2).alias('som_M_eur'), pl.col('breakeven_cac_eur').round(0), 'tier').head(10)

Ceiling €4.02bn | TAM €663M | SAM €89M | SOM €10.7M


rank,country_name,departures_M,disrupted_%,avg_amount_eur,som_M_eur,breakeven_cac_eur,tier
i64,str,f64,f64,f64,f64,f64,str
1,"""United Kingdom""",140.2,1.59,370.0,2.2,66.0,"""Core: defend and scale"""
2,"""Germany""",96.9,1.49,344.0,1.33,61.0,"""Core: defend and scale"""
3,"""France""",85.4,1.72,326.0,1.28,58.0,"""Core: defend and scale"""
4,"""Spain""",140.9,0.91,321.0,1.1,57.0,"""Core: defend and scale"""
5,"""Netherlands""",35.6,2.25,359.0,0.77,64.0,"""Next: invest in local growth"""
6,"""Italy""",97.6,1.0,287.0,0.75,51.0,"""Next: invest in local growth"""
7,"""Portugal""",32.0,1.33,350.0,0.4,63.0,"""Next: invest in local growth"""
8,"""Belgium""",14.3,2.57,333.0,0.33,59.0,"""Next: invest in local growth"""
9,"""Switzerland""",25.3,1.47,318.0,0.32,57.0,"""Long tail: SEO only"""


**Robustness.** Ranks under other ways to measure disruption and with every assumption at its low value: the top 5 does not change.

In [3]:
pl.read_csv(PUB / 'rank_stability.csv').head(8)

country,central,no_shrinkage,arrivals_only,departures_only,all_low
str,i64,i64,i64,i64,i64
"""UK""",1,1,1,1,1
"""DE""",2,2,3,2,2
"""FR""",3,3,2,3,3
"""ES""",4,4,4,4,4
"""NL""",5,5,6,5,5
"""IT""",6,6,5,6,6
"""PT""",7,7,7,7,7
"""BE""",8,8,10,9,8


In [4]:
s = pl.read_csv(PUB / 'market_sensitivity.csv')
s.with_columns((pl.col('som_total_eur')/1e6).round(1).alias('som_total_M')).select('assumption', 'case', 'value', 'som_total_M')

assumption,case,value,som_total_M
str,str,f64,f64
"""airline_control""","""low""",0.4,7.7
"""airline_control""","""high""",0.7,13.6
"""claim_rate""","""low""",0.2,6.1
"""claim_rate""","""high""",0.5,15.2
"""agency_share""","""low""",0.3,7.1
"""agency_share""","""high""",0.6,14.2
…,…,…,…
"""fee""","""low""",0.25,8.9
"""fee""","""high""",0.35,12.4


**US benchmark (BTS 2025, 7M flights).** Used to calibrate two assumptions: the share of 3h+ delays that last 3-4 h (long-haul amounts are halved)
and the share of 3h+ delay minutes caused by the carrier (a lower bound for 'within the airline's control').

In [5]:
pl.read_csv(PUB / 'us_benchmark.csv')

metric,value
str,f64
"""flights""",7.001619e6
"""cancel_rate""",0.014693
"""late_15_rate""",0.216254
"""late_3h_rate""",0.016194
"""share_3to4h_of_3h""",0.460455
"""carrier_share_of_3h_delay_minu…",0.392508


## 2. Funnel and data quality (SIMULATED)
The GA4 export only holds consented sessions; the CRM holds every claim. The daily ratio between them is the first health check.

In [6]:
run_sql_file('01_funnel_by_step.sql', con)

device_category,sessions,checks,starts,submits,signed,check_rate,start_rate,submit_rate,sign_rate,session_to_submit
str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64
"""mobile""",385543,165869,47503,27195,23355,0.4302,0.2864,0.5725,0.8588,0.0705
"""desktop""",220225,103229,35135,24592,21138,0.4687,0.3404,0.6999,0.8595,0.1117
"""tablet""",33162,14538,4439,2815,2401,0.4384,0.3053,0.6342,0.8529,0.0849


In [7]:
qa = run_sql_file('03_qa_ga4_vs_crm_daily.sql', con)
qa.filter(pl.col('status') == 'ALERT').group_by('device_category').agg(pl.len().alias('alert_days'), pl.col('d').min().alias('first_alert'))

device_category,alert_days,first_alert
str,u32,date
"""mobile""",16,2026-09-15
"""tablet""",23,2025-05-03


In [8]:
print('Consent coverage before/after the German banner:', R['tracking']['consent']['coverage']['DE'])
print('German CRM claims per week before/after:', round(R['tracking']['consent']['de_crm_weekly_before']), round(R['tracking']['consent']['de_crm_weekly_after']))
run_sql_file('05_qa_duplicate_claims.sql', con)

Consent coverage before/after the German banner: {'before': 0.627906976744186, 'after': 0.4702750665483585}
German CRM claims per week before/after: 101 101


d,duplicate_claims,example_duplicate,example_original
date,i64,str,str
2026-01-20,186,"""CLD101613""","""CL101613"""


In [9]:
pl.DataFrame(R['tracking']['platforms']['by_platform'])

platform,reported,crm_claims,spend_eur,factor
str,f64,i64,f64,f64
"""google_ads""",29958.0,26904,1.54069e6,1.113515
"""meta_ads""",15143.0,7304,380959.0,2.073248
"""microsoft_ads""",8095.0,6847,135224.0,1.18227


## 3. Testing with rigour (SIMULATED)
**Fee-display A/B test.** First the sample ratio check, then the read on what was randomised (server assignment, outcome in the CRM).

In [10]:
run_sql_file('11_ab_test_srm.sql', con)

device_category,variant,server_assignments,ga_exposures,ga_share,server_share
str,str,i64,i64,f64,f64
"""desktop""","""A""",4202,2781,0.4961,0.498
"""desktop""","""B""",4235,2825,0.5039,0.502
"""mobile""","""A""",6714,4468,0.5395,0.5003
"""mobile""","""B""",6706,3813,0.4605,0.4997
"""tablet""","""A""",578,379,0.5027,0.4928
"""tablet""","""B""",595,375,0.4973,0.5072


In [11]:
t = R['testing']
print(f"SRM on GA4 exposures: B share {t['srm']['ga_share_b']:.3f}, chi-square p = {t['srm']['ga_p']:.1e} (server log p = {t['srm']['server_p']:.2f})")
print(f"Raw GA4 read: lift {t['raw']['lift_rel']:+.1%}, p = {t['raw']['p_value']:.4f}")
print(f"Corrected (ITT, CRM): lift {t['itt']['lift_rel']:+.1%}, p = {t['itt']['p_value']:.2f}, 95% CI {t['itt']['ci_low']/t['itt']['rate_a']:+.1%} to {t['itt']['ci_high']/t['itt']['rate_a']:+.1%}")
print(f"Power: MDE with {t['power']['n_per_arm_actual']:,} per arm = {t['power']['mde_rel_actual']:.1%}; n per arm for a 5% lift = {t['power']['n_per_arm_for_5pct']:,.0f}")

SRM on GA4 exposures: B share 0.479, chi-square p = 3.7e-07 (server log p = 0.78)
Raw GA4 read: lift +9.6%, p = 0.0000
Corrected (ITT, CRM): lift -1.6%, p = 0.39, 95% CI -5.1% to +2.0%
Power: MDE with 11,494 per arm = 5.3%; n per arm for a 5% lift = 11,758


**Quasi-experiment.** Microsoft brand campaigns paused in France for 8 weeks. Outcome = weekly claims through brand searches (Microsoft brand + SEO).
Regression `log(y) ~ FR x pause + country FE + week FE`, robust (HC1) standard errors, plus a placebo pause in the pre-period.

In [12]:
run_sql_file('10_brand_pause_did.sql', con)

grp,period,weeks,ms_brand_per_week,seo_per_week,brand_demand_per_week
str,str,i64,f64,f64,f64
"""treated (FR)""","""before""",13,9.9,51.7,61.6
"""treated (FR)""","""pause""",8,0.0,52.1,52.1
"""control""","""before""",13,11.8,62.9,74.7
"""control""","""pause""",8,10.2,53.2,63.4


In [13]:
d = t['did']
print(f"DiD (level): {d['beta_level']:+.2f} claims/week, 95% CI {d['ci_level'][0]:.1f} to {d['ci_level'][1]:+.1f}; ads were credited with {d['fr_ms_brand_per_week_before']:.1f}/week")
print(f"DiD (log): {d['beta_log']:+.3f} (SE {d['se_log']:.3f}); placebo {d['placebo_beta_log']:+.3f} (SE {d['placebo_se_log']:.3f})")

DiD (level): +1.80 claims/week, 95% CI -5.0 to +8.6; ads were credited with 9.9/week
DiD (log): -0.012 (SE 0.053); placebo -0.061 (SE 0.070)


## 4. Spend, forecast and acceptance (SIMULATED)

In [14]:
pl.read_csv(DATA / 'output' / 'unit_economics_channel.csv').select('traffic_source', 'campaign_type', 'claims', 'paid', pl.col('spend').round(0), pl.col('cac_paid').round(1), pl.col('roas').round(2), 'incrementality', 'incr_source')

traffic_source,campaign_type,claims,paid,spend,cac_paid,roas,incrementality,incr_source
str,str,i64,f64,f64,f64,f64,f64,str
"""google_ads""","""non_brand""",5122,2839.0,451160.0,158.9,1.36,1.0,"""assumed 100% until tested"""
"""meta_ads""","""prospecting""",1378,698.0,96716.0,138.6,1.55,1.0,"""assumed 100% until tested"""
"""google_ads""","""brand""",3498,1917.0,39585.0,20.6,10.95,null,"""untested: run the same pause"""
"""microsoft_ads""","""non_brand""",536,327.0,29737.0,90.9,2.21,1.0,"""assumed 100% until tested"""
"""meta_ads""","""retargeting""",920,442.0,23254.0,52.6,4.18,1.0,"""assumed 100% until tested"""
"""microsoft_ads""","""brand""",1648,919.0,13620.0,14.8,14.68,0.0,"""measured (France pause)"""
"""crm""","""email""",5513,3065.0,0.0,0.0,inf,null,""""""
"""organic_social""","""organic""",877,462.0,0.0,0.0,inf,null,""""""
"""seo""","""organic""",9002,5013.0,0.0,0.0,inf,null,""""""


In [15]:
print(R['spend']['waste'])
print(R['spend']['forecast'])
pl.read_csv(DATA / 'output' / 'budget_6m.csv')

{'ms_brand_spend_12m': 28879.25000000007, 'google_brand_spend_12m': 85514.45999999998, 'incrementality': 0.0, 'ms_brand_waste_12m': 28879.25000000007, 'total_spend_12m': 1402800.8599999996, 'note': 'Google brand not tested: same test recommended before cutting it'}
{'total_6m': 27797.60099233262, 'low_6m': 25784.885548198756, 'high_6m': 29974.490320934703, 'mape': 0.03953106302306594, 'budget_current_6m': 628842.0243430719, 'budget_proposed_6m': 615896.1483108106}


market,forecast,s,cac,paid_claims,budget_current,budget_proposed
str,f64,f64,f64,f64,f64,f64
"""UK""",9145.111982,0.449487,50.324005,4110.611636,206862.441187,202603.79527
"""DE""",5896.728359,0.45461,50.002419,2680.711855,134042.076735,131282.572692
"""FR""",4670.032258,0.445433,50.853351,2080.186877,105784.473376,103606.704357
"""ES""",4634.171735,0.452459,49.330841,2096.772785,103435.565684,101306.153274
"""IT""",3451.556658,0.457271,49.874946,1578.2968,78717.467361,77096.922717


In [16]:
print('Summer peak check:', R['spend']['season'])
pl.read_csv(DATA / 'output' / 'acceptance_change_by_carrier.csv').head(4)

Summer peak check: {'jul26_vs_jun26': 0.3018619522287005, 'jul26_yoy': 0.176, 'yoy_min': 0.148, 'yoy_max': 0.206, 'jul25_vs_jun25': 0.31780515117581176}


carrier,before,after,change_pts,p_value,n_after
str,f64,f64,f64,f64,i64
"""Carrier H""",0.629661,0.298864,-0.330797,0.0,1760
"""Carrier C""",0.570029,0.535876,-0.034153,0.019734,2648
"""Carrier E""",0.504091,0.475082,-0.029009,0.079467,2147
"""Carrier A""",0.669837,0.64176,-0.028077,0.02484,3386


## 5. Subscription pricing (PUBLIC + SIMULATED)
Expected cost per subscriber = trips x 2 flights x real eligible disruption per flight (public model) x cost of a covered claim
(handling, plus the commission that would have been earned anyway). Then the pilot: three random price tiers.

In [17]:
pl.read_csv(DATA / 'output' / 'subscription_cost_curve.csv').select('trips_per_year', pl.col('expected_claims').round(3), pl.col('cost_incremental_eur').round(1), pl.col('cost_full_cannibalisation_eur').round(1))

trips_per_year,expected_claims,cost_incremental_eur,cost_full_cannibalisation_eur
i64,f64,f64,f64
1,0.02,0.7,3.1
2,0.04,1.4,6.1
3,0.059,2.1,9.2
4,0.079,2.8,12.3
6,0.119,4.2,18.4
8,0.159,5.6,24.6
10,0.198,6.9,30.7
12,0.238,8.3,36.8
15,0.297,10.4,46.1


In [18]:
pl.read_csv(DATA / 'output' / 'subscription_tiers.csv').select('tier', 'price_eur', 'take_up', 'trips_subscribers', pl.col('cost_per_sub_full').round(1), pl.col('margin_per_sub_year').round(1), pl.col('renewal_projected').round(2), pl.col('ltv_per_offer').round(2), 'share_subs_above_breakeven_trips')

tier,price_eur,take_up,trips_subscribers,cost_per_sub_full,margin_per_sub_year,renewal_projected,ltv_per_offer,share_subs_above_breakeven_trips
str,f64,f64,f64,f64,f64,f64,f64,f64
"""Lite""",29.0,0.0947,6.67,20.5,15.6,0.47,2.6,0.208731
"""Plus""",49.0,0.0543,6.89,21.2,34.6,0.31,2.54,0.095012
"""Pro""",79.0,0.0284,8.51,26.1,60.6,0.31,2.21,0.068966


## 6. Detection scorecard (run last)
Only now is the hidden answer key (`data/_hidden_truth.csv`) read, to score the checks above.

In [19]:
pl.read_csv(DATA / 'output' / 'detection_scorecard.csv').select('id', 'event', 'detected', 'detected_at', 'method')

id,event,detected,detected_at,method
i64,str,bool,str,str
1,"""broken_tag""",true,"""2026-09-15""","""03 daily GA4/CRM ratio, 3 aler…"
2,"""consent_mode""",true,"""2026-02-16""","""04 weekly coverage, 3-sd drop,…"
3,"""double_counting""",true,"""all period""","""06 platform conversions vs CRM…"
4,"""brand_cannibalisation""",true,"""2026-02-02""","""10 + regression DiD"""
5,"""ab_srm""",true,"""2026-06-01""","""11 chi-square on exposures vs …"
6,"""acceptance_drop""",true,"""2026-05""","""08 + two-proportion z-test"""
7,"""seasonal_peak""",true,"""2026-07""","""12 YoY growth stable while mon…"
8,"""duplicates""",true,"""2026-01-20""","""05 LAG window, same key within…"


## Conclusions
1. Fix the mobile `claim_submit` tag before reading any funnel test; the CRM, not GA4, is the source for claims.
2. Microsoft brand ads in France bought claims SEO would have brought: stop them, and test Google brand the same way.
3. UK, Germany, France and Spain are the core; the Netherlands is the next market in every sensitivity case.
4. Launch the subscription at €49 a year with a trip cap: subscribers fly far more than average.